# AB-SAW 占有率ごとの抵抗解析

目標占有率 0.10〜0.70 を 0.01 刻みで変え、各条件で1試料を新規生成して解析します。既定の格子サイズ・腕長・seed・電極条件は前回の単一試料と同じです。全セルを上から実行してください。

実測占有率は分子数への換算や配置上限により目標値と異なります。両電極に接点があり電極間が非連結のときは `R_eff/R_0=inf`、電極の接点がない場合は空欄になります。配置未達・実行エラーも状態を残します。

## 1. Driveに接続し、GitHubの生成・解析コードを取得

In [ ]:
from pathlib import Path
import csv
import json
import math
import subprocess
import sys
from collections import Counter
from datetime import datetime, timezone
from uuid import uuid4

from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/tetra-peg-network/resistance_analysis")
if not drive_root.is_dir():
    raise FileNotFoundError(drive_root)

revision = "56a533de53ea65ff5707050c309deec40349a3b8"
repo_url = "https://github.com/takakei510/tetra-peg-network.git"
repo = Path("/content") / f"ab-saw-resistance-{uuid4().hex[:8]}"
subprocess.run(["git", "init", "-q", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", repo_url, revision], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["make", "build/demo_network"], cwd=repo, check=True)
generator = repo / "build/demo_network"

try:
    import numpy
    import scipy
    import matplotlib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy", "scipy", "matplotlib"], check=True)

## 2. 実験条件と再開設定

In [ ]:
# 10 → 0.10、70 → 0.70。整数指定なので 0.01 刻みに丸め誤差は入りません。
PHI_START, PHI_STOP = 10, 70
L, SEED, ARM_LENGTH, A_PROBABILITY = 32, 3, 5, 0.5
MAX_PLACEMENT_ATTEMPTS = 1_000_000
ANALYSIS_SETTINGS = {
    "axis": "x",
    "electrode_layers": 1,
    "branch_resistance": 1.0,
    "voltage_high": 1.0,
    "voltage_low": 0.0,
}

# 中断した実験を再開する場合は、表示された実験フォルダ名を指定します。
RESUME_RUN_ID = ""
run_id = RESUME_RUN_ID or (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
)
if Path(run_id).name != run_id or run_id in {".", ".."}:
    raise ValueError("実験フォルダ名だけを指定してください。")
sweep_dir = drive_root / "occupancy_sweeps" / run_id
print("出力先", sweep_dir)

## 3. Cで試料を生成し、Pythonで抵抗を計算

各点の生成CSV・枝電流・電位をDriveに保存し、集計CSVは1条件ごとに更新します。中断時は手順2の `RESUME_RUN_ID` に表示済みの実験フォルダ名を入れ、セルを再実行できます。

In [ ]:
if not 0 < PHI_START <= PHI_STOP <= 100:
    raise ValueError("目標占有率の範囲を 1〜100 の整数で指定してください。")

def read_single_row(path):
    with path.open(encoding="utf-8", newline="") as stream:
        row, = csv.DictReader(stream)
    return row

def write_json(path, data):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temp.replace(path)

def write_summary(rows):
    fields = list(dict.fromkeys(key for row in rows for key in row))
    temp = sweep_dir / "sweep-summary.tmp"
    with temp.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temp.replace(sweep_dir / "sweep-summary.csv")

metadata = {
    "code_commit": revision, "start_percent": PHI_START, "stop_percent": PHI_STOP,
    "step_percent": 1, "box_size": L, "seed": SEED, "arm_length": ARM_LENGTH,
    "a_probability": A_PROBABILITY, "max_placement_attempts": MAX_PLACEMENT_ATTEMPTS,
    "analysis": ANALYSIS_SETTINGS,
}
manifest = sweep_dir / "sweep-run.json"
if RESUME_RUN_ID:
    if json.loads(manifest.read_text(encoding="utf-8")) != metadata:
        raise ValueError("再開時の設定は最初の実行と一致させてください。")
else:
    sweep_dir.mkdir(parents=True, exist_ok=False)
    write_json(manifest, metadata)
settings_path = sweep_dir / "resistance-settings.json"
write_json(settings_path, ANALYSIS_SETTINGS)

csv_names = ("candidates", "bonds", "cluster-members", "cluster-summary",
             "network-summary", "placement-summary")
rows = []
for percent in range(PHI_START, PHI_STOP + 1):
    phi = f"{percent / 100:.2f}"
    case = sweep_dir / f"phi_{phi}"
    outcome = case / "case-summary.json"
    if outcome.is_file():
        rows.append(json.loads(outcome.read_text(encoding="utf-8")))
        write_summary(rows)
        continue

    # 中断や実行エラーの記録を残し、新しい試行フォルダから再開します。
    attempt = case / ("attempt_" + uuid4().hex[:8])
    sample = attempt / "sample"
    sample.mkdir(parents=True)
    config = sample / "config.cfg"
    config.write_text(
        f"dim = 3\nL = {L}\nseed = {SEED}\ntarget_phi = {phi}\n"
        f"arm_length = {ARM_LENGTH}\na_probability = {A_PROBABILITY}\n",
        encoding="utf-8",
    )
    command = [str(generator), str(config),
               *(str(sample / f"{name}.csv") for name in csv_names),
               str(MAX_PLACEMENT_ATTEMPTS)]
    with (sample / "trajectories.csv").open("wb") as trajectory, (sample / "generation.log").open("wb") as log:
        generation = subprocess.run(command, stdout=trajectory, stderr=log)
    placement_path = sample / "placement-summary.csv"
    placement = read_single_row(placement_path) if placement_path.is_file() else {}
    result = {"target_phi": phi, "actual_phi": placement.get("actual_phi"),
              "L": L, "seed": SEED, "placed_molecules": placement.get("placed_molecules"),
              "placement_attempts": placement.get("attempts"),
              "stop_reason": placement.get("stop_reason"),
              "attempt_dir": attempt.relative_to(sweep_dir).as_posix()}
    if generation.returncode == 3 and placement.get("completed") == "0":
        result["status"] = "generation_incomplete"
    elif generation.returncode != 0 or placement.get("completed") != "1":
        result["status"] = "generation_error"
    else:
        write_json(sample / "run.json", {
            "status": "completed", "run_id": f"{run_id}_phi_{phi}", "git_commit": revision,
        })
        result_dir = attempt / "resistance"
        with (attempt / "analysis.log").open("wb") as log:
            analysis = subprocess.run(
                [sys.executable, str(repo / "scripts/analysis/analyze_resistance.py"), str(sample),
                 "--settings", str(settings_path), "--output-dir", str(result_dir)],
                cwd=repo, stdout=log, stderr=subprocess.STDOUT,
            )
        result.update(read_single_row(result_dir / "resistance-summary.csv")
                      if analysis.returncode == 0 else {"status": "analysis_error"})
    write_json(attempt / "case-summary.json", result)
    if result["status"] not in {"generation_error", "analysis_error"}:
        write_json(outcome, result)
    rows.append(result)
    write_summary(rows)
    print(phi, result["status"], result.get("resistance_ratio", ""), flush=True)

print(dict(Counter(row["status"] for row in rows)))

## 4. 図を確認

上段では電極間が非連結の試料を導電度ゼロで示します。下段には有限の抵抗だけを表示します。

In [ ]:
import matplotlib.pyplot as plt

phi = [float(row["target_phi"]) for row in rows]
def ordinate(row, field, valid_status):
    return float(row[field]) if row["status"] in valid_status else math.nan

fig, (conductance, resistance) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, layout="constrained")
conductance.plot(phi, [ordinate(row, "effective_conductance", {"connected", "disconnected"})
                       * ANALYSIS_SETTINGS["branch_resistance"] for row in rows], ".-")
resistance.plot(phi, [ordinate(row, "resistance_ratio", {"connected"}) for row in rows], ".-")
conductance.set_ylabel(r"$G_{\mathrm{eff}} R_0$")
resistance.set_ylabel(r"$R_{\mathrm{eff}} / R_0$")
resistance.set_xlabel(r"Target occupancy $\phi$")
conductance.set_title(f"AB-SAW occupancy sweep  |  L={L}, seed={SEED}")
for ax in (conductance, resistance):
    ax.grid(alpha=0.25)
fig.savefig(sweep_dir / "resistance-vs-occupancy.png", dpi=180)
plt.show()
print("集計CSV", sweep_dir / "sweep-summary.csv")